# 01 - Limpeza e preparação

Case Olist | Tech Challenge Fase 1

Este notebook prepara a base que todas as análises do projeto usam. Ele carrega as tabelas, avalia a
qualidade dos dados, padroniza os tipos e monta o modelo dimensional.

Uma decisão de projeto: ele não aplica nenhum filtro de análise. Não corta período e não remove status de
pedido. Cada análise tem um recorte diferente, e quem estuda cancelamento precisa dos pedidos que o cálculo
de receita descarta. Os filtros ficam nos notebooks seguintes, sempre com a justificativa ao lado.

Entrada: `data/raw/` | Saída: `data/processed/`

In [1]:
import os
import pandas as pd

pd.set_option('display.max_columns', None)
os.makedirs('../data/processed', exist_ok=True)

In [2]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
cat_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

tabelas = {
    'customers': customers, 'order_items': order_items, 'payments': payments,
    'reviews': reviews, 'orders': orders, 'products': products,
    'sellers': sellers, 'cat_translation': cat_translation,
}

for nome, tabela in tabelas.items():
    print(nome, tabela.shape)

customers (99441, 5)
order_items (112650, 7)
payments (103886, 5)
reviews (99224, 7)
orders (99441, 8)
products (32951, 9)
sellers (3095, 4)
cat_translation (71, 2)


Deixamos `geolocation` fora do escopo. Ela repete coordenadas por prefixo de CEP e as análises usam a UF
do cliente, que já vem em `customers`.

## Qualidade dos dados

In [3]:
for nome, tabela in tabelas.items():
    print(nome.ljust(18), 'nulos:', str(tabela.isnull().sum().sum()).ljust(8),
          'duplicadas:', tabela.duplicated().sum())

customers          nulos: 0        duplicadas: 0
order_items        nulos: 0        duplicadas: 0
payments           nulos: 0        duplicadas: 0
reviews            nulos: 145903   duplicadas: 0
orders             nulos: 4908     duplicadas: 0
products           nulos: 2448     duplicadas: 0


sellers            nulos: 0        duplicadas: 0
cat_translation    nulos: 0        duplicadas: 0


In [4]:
for nome in ['orders', 'products', 'reviews']:
    print('---', nome, '---')
    nulos = tabelas[nome].isnull().sum()
    print(nulos[nulos > 0])
    print()

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64



In [5]:
print('nulos em price:', order_items['price'].isnull().sum())
print('nulos em freight_value:', order_items['freight_value'].isnull().sum())
print('nulos em order_purchase_timestamp:', orders['order_purchase_timestamp'].isnull().sum())

nulos em price: 0
nulos em freight_value: 0


nulos em order_purchase_timestamp: 0


Nenhuma tabela tem linha inteira duplicada e nenhuma coluna usada em cálculo de valor tem nulo.

Os nulos que existem são esperados e não foram preenchidos: em `orders` são datas de entrega de pedidos que
não foram entregues, em `reviews` são comentários opcionais, em `products` são medidas físicas. A única
exceção tratada é a categoria do produto: duas categorias não constam na tabela de tradução e mantêm o
nome original em português.

## Datas

A padronização das datas revelou um detalhe que altera resultado, então ele fica registrado aqui e vale para
todo o projeto.

In [6]:
colunas_data = [
    'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
    'order_delivered_customer_date', 'order_estimated_delivery_date',
]

for col in colunas_data:
    orders[col] = pd.to_datetime(orders[col])

print('data prometida sempre à meia-noite?',
      (orders['order_estimated_delivery_date'].dt.time.astype(str) == '00:00:00').all())
print()
print(orders[['order_delivered_customer_date', 'order_estimated_delivery_date']].dropna().head(3))

data prometida sempre à meia-noite? True

  order_delivered_customer_date order_estimated_delivery_date
0           2017-10-10 21:25:13                    2017-10-18
1           2018-08-07 15:27:45                    2018-08-13
2           2018-08-17 18:06:29                    2018-09-04


In [7]:
entregues = orders[orders['order_delivered_customer_date'].notna()]

com_hora = (entregues['order_delivered_customer_date'] > entregues['order_estimated_delivery_date'])
so_data = (entregues['order_delivered_customer_date'].dt.normalize() >
           entregues['order_estimated_delivery_date'].dt.normalize())

print('atraso comparando data e hora:', round(com_hora.mean() * 100, 2), '%')
print('atraso comparando só a data: ', round(so_data.mean() * 100, 2), '%')
print('pedidos entregues no dia prometido, mas depois da meia-noite:', (com_hora & ~so_data).sum())

atraso comparando data e hora: 8.11 %
atraso comparando só a data:  6.77 %
pedidos entregues no dia prometido, mas depois da meia-noite: 1292


A data prometida vem sem hora, sempre meia-noite. A data real vem com hora cheia. Comparar as duas
diretamente classifica como atrasado um pedido entregue no dia certo às 15h, o que infla o atraso da base de
6,8% para 8,1%.

Critério adotado no projeto: atraso é a data da entrega posterior à data prometida, comparando apenas o dia.
A coluna sai pronta daqui para não ser recalculada em cada análise.

O `dias_entrega` conta da compra até o recebimento, não do envio. Ele mede a espera do cliente, que inclui
o tempo de aprovação do pagamento, e não apenas o tempo de transporte.

In [8]:
orders = orders.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state']],
    on='customer_id', how='left'
)

orders['atrasou'] = (orders['order_delivered_customer_date'].dt.normalize() >
                     orders['order_estimated_delivery_date'].dt.normalize())
orders['dias_entrega'] = (orders['order_delivered_customer_date'] -
                          orders['order_purchase_timestamp']).dt.days
orders['mes'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

print('pedidos sem cliente correspondente:', orders['customer_unique_id'].isnull().sum())
print('pedidos com data de entrega anterior à compra:', (orders['dias_entrega'] < 0).sum())

pedidos sem cliente correspondente: 0
pedidos com data de entrega anterior à compra: 0


## Panorama da base

Registramos os números completos aqui, sem filtrar, para servir de referência às análises seguintes.

In [9]:
print('período:', orders['order_purchase_timestamp'].min(), 'a', orders['order_purchase_timestamp'].max())
print('total de pedidos:', len(orders))
print()
print(orders['order_status'].value_counts())

período: 2016-09-04 21:15:19 a 2018-10-17 17:30:18
total de pedidos: 99441

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [10]:
print(orders['mes'].value_counts().sort_index())

mes
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Name: count, dtype: int64


In [11]:
for status in ['canceled', 'unavailable', 'shipped', 'processing', 'invoiced', 'delivered']:
    do_status = orders[orders['order_status'] == status]
    com_item = do_status['order_id'].isin(order_items['order_id']).mean() * 100
    print(status.ljust(12), 'pedidos:', str(len(do_status)).ljust(8), 'com produto:', round(com_item, 1), '%')

canceled     pedidos: 625      com produto: 73.8 %
unavailable  pedidos: 609      com produto: 1.0 %


shipped      pedidos: 1107     com produto: 99.9 %
processing   pedidos: 301      com produto: 100.0 %
invoiced     pedidos: 314      com produto: 99.4 %
delivered    pedidos: 96478    com produto: 100.0 %


Dois pontos que cada análise vai precisar decidir.

As pontas da série são fracas: set/2016 tem 4 pedidos, out/2016 tem 324, nov/2016 não existe e dez/2016 tem 1.
No fim, set e out de 2018 somam 20 pedidos.

E parte dos pedidos não virou venda. Apenas 1,0% dos `unavailable` tem produto associado e 73,8% dos `canceled`
têm, contra 99% ou mais nos demais status. Não excluímos nada aqui, apenas deixamos a evidência registrada.

## Modelo dimensional

Tabela fato no grão de item de pedido, com dimensões de cliente, produto, vendedor e data.

Antes dos merges, `payments` e `reviews` precisam ser agregados por pedido. As duas têm mais de uma linha por
`order_id`, e sem agregar cada item viraria várias linhas com a receita contada em dobro.

As colunas de pagamento e de avaliação entram na tabela fato mas não são usadas nos notebooks 02 e 03. Elas
ficam disponíveis porque esta base é compartilhada e outras frentes do projeto podem precisar delas.

In [12]:
print('payments - linhas:', len(payments), '| pedidos:', payments['order_id'].nunique())
print('reviews  - linhas:', len(reviews), '| pedidos:', reviews['order_id'].nunique())

payments - linhas: 103886 | pedidos: 99440
reviews  - linhas: 99224 | pedidos: 98673


In [13]:
pagamento_por_pedido = payments.groupby('order_id')['payment_value'].sum().reset_index()
pagamento_por_pedido.columns = ['order_id', 'total_pago']

tipo_pagamento = payments.sort_values('payment_value', ascending=False).groupby('order_id').first().reset_index()
tipo_pagamento = tipo_pagamento[['order_id', 'payment_type', 'payment_installments']]

nota_por_pedido = reviews.groupby('order_id')['review_score'].mean().reset_index()

dim_produtos = products.merge(cat_translation, on='product_category_name', how='left')
# duas categorias nao constam na tabela de traducao. Mantemos o nome original
# em vez de agrupar num rotulo 'unknown', que nao e uma categoria real.
dim_produtos['product_category_name_english'] = dim_produtos['product_category_name_english'].fillna(
    dim_produtos['product_category_name'])

In [14]:
fato_itens = order_items.merge(orders, on='order_id', how='left')
fato_itens = fato_itens.merge(pagamento_por_pedido, on='order_id', how='left')
fato_itens = fato_itens.merge(tipo_pagamento, on='order_id', how='left')
fato_itens = fato_itens.merge(nota_por_pedido, on='order_id', how='left')
fato_itens = fato_itens.merge(dim_produtos[['product_id', 'product_category_name_english']], on='product_id', how='left')
fato_itens = fato_itens.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')

fato_itens['receita'] = fato_itens['price'] + fato_itens['freight_value']

print('tabela fato:', fato_itens.shape)

tabela fato: (112650, 26)


In [15]:
print('linhas em order_items:', len(order_items))
print('linhas na tabela fato:', len(fato_itens))
print('sem duplicação?', len(fato_itens) == len(order_items))
print()
print('soma de price na origem:', round(order_items['price'].sum(), 2))
print('soma de price na fato  :', round(fato_itens['price'].sum(), 2))
print()
print('itens sem categoria:', fato_itens['product_category_name_english'].isnull().sum())
print('itens sem estado do cliente:', fato_itens['customer_state'].isnull().sum())

linhas em order_items: 112650
linhas na tabela fato: 112650
sem duplicação? True

soma de price na origem: 13591643.7
soma de price na fato  : 13591643.7

itens sem categoria: 1603
itens sem estado do cliente: 0


As três validações passam: a contagem de linhas não mudou depois de seis merges, a soma de preço bate
com a origem e nenhuma chave ficou órfã. Se qualquer uma falhar, há duplicação em algum merge e os números
seguintes estarão errados.

In [16]:
dim_data = orders[['order_id', 'order_purchase_timestamp', 'mes']].copy()
dim_data['ano'] = dim_data['order_purchase_timestamp'].dt.year
dim_data['dia_semana'] = dim_data['order_purchase_timestamp'].dt.dayofweek

fato_itens.to_csv('../data/processed/fato_itens.csv', index=False)
orders.to_csv('../data/processed/pedidos_tratados.csv', index=False)
dim_produtos.to_csv('../data/processed/dim_produtos.csv', index=False)
customers.to_csv('../data/processed/dim_clientes.csv', index=False)
sellers.to_csv('../data/processed/dim_vendedores.csv', index=False)
dim_data.to_csv('../data/processed/dim_data.csv', index=False)

print('base salva em data/processed/')

base salva em data/processed/


## Decisões registradas

| Decisão | Critério |
|---|---|
| Receita = preço do produto + frete | valor transacionado no item |
| Atraso comparado só pela data | a data prometida não tem hora, e comparar com hora infla 1,3 ponto |
| `payments` e `reviews` agregados por pedido | evita duplicar receita nos merges |
| Categoria sem tradução mantém o nome em português | não cria um rótulo `unknown` que não é categoria |
| Nenhum filtro de período ou status | é responsabilidade de cada análise, com justificativa |
| `geolocation` fora do escopo | repetição alta e a UF já vem em `customers` |

O notebook 02 analisa crescimento e receita. O 03 investiga a concentração geográfica e a entrega.